In [66]:
import sys

# Remove all cached models
for key in list(sys.modules.keys()):
    if key.startswith("models"):
        del sys.modules[key]

# imports
from models.content      import Movie, TVSeries, Documentary
from models.subscription import SubscriptionPlan, Subscription, PlanTier, SubscriptionStatus
from models.payment      import Payment, Invoice, PaymentMethod, PaymentStatus
from models.user         import RegularUser, Administrator, ViewingRecord, Notification, Feedback
from models.catalogue    import ContentCatalogue, StreamingService
from datetime            import date, timedelta

print("All classes imported successfully!")
print("  Content      -> Movie, TVSeries, Documentary")
print("  Subscription -> SubscriptionPlan, Subscription")
print("  Payment      -> Payment, Invoice")
print("  User         -> RegularUser, Administrator")
print("  Services     -> ContentCatalogue, StreamingService")

All classes imported successfully!
  Content      -> Movie, TVSeries, Documentary
  Subscription -> SubscriptionPlan, Subscription
  Payment      -> Payment, Invoice
  User         -> RegularUser, Administrator
  Services     -> ContentCatalogue, StreamingService


In [67]:
 # Section header for helper functions

def make_plans():  # Function to create and return subscription plans
    """Return Basic, Standard, Premium plan objects."""  # Function description
    basic    = SubscriptionPlan("plan-B", PlanTier.BASIC,    8.99,  1, False, False, "SD quality, 1 screen.")  # Create Basic plan object
    standard = SubscriptionPlan("plan-S", PlanTier.STANDARD, 13.99, 2, True,  False, "HD quality, 2 screens.")  # Create Standard plan object
    premium  = SubscriptionPlan("plan-P", PlanTier.PREMIUM,  17.99, 4, True,  True,  "4K quality, 4 screens + all premium content.")  # Create Premium plan object
    return basic, standard, premium  # Return all three plans as a tuple


def make_catalogue():  # Function to create and return a pre-filled content catalogue
    """Return a ContentCatalogue pre-loaded with sample content."""  # Function description
    cat = ContentCatalogue()  # Create a new ContentCatalogue object

    cat.add_content(Movie("m001", "Inception",         "Sci-Fi",  2010, 148, director="Nolan",      rating=8.8))
    # Add a Movie object (Inception) to catalogue

    cat.add_content(Movie("m002", "Dune: Part Two",    "Sci-Fi",  2024, 166, director="Villeneuve", rating=8.5, is_premium=True, price=5.99))
    # Add a premium Movie (requires purchase)

    cat.add_content(TVSeries("t001", "Breaking Bad",   "Drama",   2008, 47,  seasons=5, episodes=62, is_completed=True, rating=9.5))
    # Add a TV series (non-premium)

    cat.add_content(TVSeries("t002", "House of Dragon","Fantasy", 2022, 60,  seasons=2, episodes=18, rating=8.1, is_premium=True, price=3.99))
    # Add a premium TV series

    cat.add_content(Documentary("d001","Planet Earth III","Nature",2023, 55, subject="Wildlife", narrator="Attenborough", rating=9.2))
    # Add a documentary content

    return cat  # Return the filled catalogue


SEP = "\n" + "─" * 58 + "\n"  # Separator line used for clean console output formatting



In [68]:
print(SEP + "TEST 1: User Account Creation")  # Print test header with separator

# Example 1 — Mohammed Al Rashid (Regular User)
u1 = RegularUser("u001", "Mohammed Al Rashid",
                 "mohammed.alrashid@streamflix.ae", "pw_001")  # Create a RegularUser object

print("\nExample 1 — Regular user:")  # Print section title
print(u1.display_info())  # Display user information

assert u1.get_name()      == "Mohammed Al Rashid"  # Check if name is correct
assert u1.get_is_active() is True  # Check if user is active
print("   Name and active status verified.")  # Confirmation message

# Example 2 — Amira Hassan (Administrator)
admin = Administrator("a001", "Amira Hassan",
                      "amira.hassan@streamflix.ae",
                      "admin_001", admin_level=2, department="Content")
# Create an Administrator object with level 2 and Content department

print("\nExample 2 — Administrator:")  # Print section title
print(admin.display_info())  # Display admin information

assert admin.get_role()        == "Administrator"  # Verify role is Administrator
assert admin.get_admin_level() == 2  # Verify admin level is 2
print("   Role and admin level verified.")  # Confirmation message

# Exception: bad email
try:
    RegularUser("u999", "Bad User", "no-at-sign", "pw")  # Attempt to create user with invalid email
except ValueError as e:  # Catch ValueError exception
    print(f"\n   Caught bad email: {e}")  # Print confirmation with error message

print("\nTEST 1 PASSED ")  # Print final success message


──────────────────────────────────────────────────────────
TEST 1: User Account Creation

Example 1 — Regular user:
[RegularUser] Mohammed Al Rashid <mohammed.alrashid@streamflix.ae>
  ID: u001 | Status: Active | Joined: 2026-03-29
  Subscription: None | History: 0 items
   Name and active status verified.

Example 2 — Administrator:
[Administrator] Amira Hassan <amira.hassan@streamflix.ae>
  ID: a001 | Status: Active | Joined: 2026-03-29
  Admin Level: 2 | Department: Content
   Role and admin level verified.

   Caught bad email: Invalid email: no-at-sign

TEST 1 PASSED 


In [69]:
print(SEP + "TEST 2: Content Catalogue Management")  # Display test title with separator

cat = ContentCatalogue()  # Initialize an empty content catalogue object

# Example 1 — Add and update a movie  # Section header
m = Movie("m010", "Oppenheimer", "Drama", 2023, 180, director="Nolan", rating=8.9)  # Create a Movie instance

cat.add_content(m)  # Add the movie object to the catalogue

found = cat.get_content_by_id("m010")  # Retrieve the movie using its ID

print(f"\nExample 1 — Added: {found.get_title()}")  # Print the title of the added movie

found.set_title("Oppenheimer (Director's Cut)")  # Modify the movie title
found.set_rating(9.1)  # Update the movie rating

print(f"  Updated title : {found.get_title()}")  # Display updated title
print(f"  Updated rating: {found.get_rating()}")  # Display updated rating

assert cat.get_content_by_id("m010").get_title() == "Oppenheimer (Director's Cut)"  # Verify updated title is correct

print("   Add and update verified.")  # Print success message for Example 1

# Example 2 — Add documentary then remove  # Section header
doc = Documentary("d010", "Seaspiracy", "Environment", 2021, 89, subject="Fishing")  # Create a Documentary object

cat.add_content(doc)  # Add documentary to catalogue

assert len(cat) == 2  # Ensure catalogue now has 2 items

cat.remove_content("d010")  # Remove documentary by ID

assert len(cat) == 1  # Confirm catalogue size decreased after removal

print(f"\nExample 2 — After remove, catalogue size: {len(cat)}")  # Display updated catalogue size
print("   Remove verified.")  # Print success message for Example 2

# Exception: duplicate ID  # Testing duplicate content addition
try:
    cat.add_content(Movie("m010", "Copy", "Drama", 2020, 90))  # Attempt to add duplicate ID
except ValueError as e:  # Catch ValueError exception
    print(f"\n   Caught duplicate ID: {e}")  # Print confirmation of caught error

# Exception: ID not found  # Testing invalid content retrieval
try:
    cat.get_content_by_id("FAKE")  # Attempt to fetch non-existent content
except KeyError as e:  # Catch KeyError exception
    print(f"   Caught missing ID: {e}")  # Print confirmation of caught error

print("\nTEST 2 PASSED ")  # Final message indicating test success


──────────────────────────────────────────────────────────
TEST 2: Content Catalogue Management
  [Catalogue] Added: 'Oppenheimer'

Example 1 — Added: Oppenheimer
  Updated title : Oppenheimer (Director's Cut)
  Updated rating: 9.1
   Add and update verified.
  [Catalogue] Added: 'Seaspiracy'
  [Catalogue] Removed: 'Seaspiracy'

Example 2 — After remove, catalogue size: 1
   Remove verified.

   Caught duplicate ID: Content ID 'm010' already exists.
   Caught missing ID: "Content 'FAKE' not found."

TEST 2 PASSED 


In [70]:
print(SEP + "TEST 3: Subscription Plan Management")  # Print a header for Test 3 with a separator

basic, standard, premium = make_plans()  # Create three subscription plans: Basic, Standard, Premium

# Example 1 — Umar Al Farouq subscribes then upgrades
u = RegularUser("u003", "Umar Al Farouq",
                "umar.alfarouq@streamflix.ae", "pw_003")  # Create a RegularUser object for Umar

sub = Subscription("sub-001", basic, date.today(), duration_days=30)  # Create a new subscription on the Basic plan for 30 days
u.subscribe(sub)  # Assign the subscription to Umar
print("\nExample 1 — Umar Al Farouq on Basic plan:")  # Print example description
print(u.get_subscription().display_info())  # Display current subscription information
assert u.has_active_subscription() is True  # Ensure the subscription is active
u.get_subscription().upgrade(premium)  # Upgrade Umar's subscription to Premium
assert u.get_subscription().get_plan().get_tier() == PlanTier.PREMIUM  # Verify the upgrade to Premium
print("   Subscribed and upgraded to Premium.")  # Print success message

# Example 2 — Fatima Al Zaabi downgrades then cancels
u2 = RegularUser("u004", "Fatima Al Zaabi",
                 "fatima.alzaabi@streamflix.ae", "pw_004")  # Create a RegularUser object for Fatima
sub2 = Subscription("sub-002", premium)  # Create a Premium subscription for Fatima
u2.subscribe(sub2)  # Assign the subscription to Fatima
u2.get_subscription().upgrade(standard)  # Downgrade subscription from Premium to Standard
assert u2.get_subscription().get_plan().get_tier() == PlanTier.STANDARD  # Verify the downgrade
u2.cancel_subscription()  # Cancel Fatima's subscription
assert u2.get_subscription().get_status() == SubscriptionStatus.CANCELLED  # Ensure subscription status is Cancelled
assert u2.has_active_subscription() is False  # Verify user no longer has an active subscription
print("\nExample 2 — Fatima Al Zaabi downgraded then cancelled.")  # Print example description
print("   Downgrade and cancel verified.")  # Print success message

# Auto-expiry check
expired_sub = Subscription("sub-old", basic,
                            start_date=date.today() - timedelta(days=60),
                            duration_days=30)  # Create a subscription that started 60 days ago with 30-day duration
assert expired_sub.is_active() is False  # Check that the subscription is no longer active
assert expired_sub.get_status() == SubscriptionStatus.EXPIRED  # Verify the status is Expired
print("\n   Auto-expiry detected correctly.")  # Print success message

print("\nTEST 3 PASSED")  # Print final confirmation that Test 3 passed


──────────────────────────────────────────────────────────
TEST 3: Subscription Plan Management

Example 1 — Umar Al Farouq on Basic plan:
Subscription [sub-001]
  Plan: Basic | Status: Active
  Valid: 2026-03-29 to 2026-04-28 | Auto-Renew: True
   Subscribed and upgraded to Premium.

Example 2 — Fatima Al Zaabi downgraded then cancelled.
   Downgrade and cancel verified.

   Auto-expiry detected correctly.

TEST 3 PASSED


In [71]:
print(SEP + "TEST 4: Streaming Content Access")  # Print a header for Test 4 with a separator

_, standard, premium_plan = make_plans()  # Create subscription plans; ignore Basic, keep Standard and Premium
cat     = make_catalogue()  # Create a content catalogue with movies and shows
service = StreamingService(cat)  # Initialize the streaming service with the catalogue

# Example 1 — Khalid (Standard) streams free content
u1 = RegularUser("u005", "Khalid Al Mansoori",
                 "khalid.almansoori@streamflix.ae", "pw_005")  # Create a RegularUser object for Khalid
u1.subscribe(Subscription("sub-010", standard))  # Subscribe Khalid to Standard plan
result = service.stream_content(u1, "m001")  # Attempt to stream free content "m001" for Khalid
print(f"\nExample 1: {result}")  # Print streaming result
assert "Now streaming" in result  # Verify streaming was successful
print("   Khalid streams Inception (free content).")  # Print success message

# Standard user blocked from premium
try:
    service.stream_content(u1, "m002")  # Khalid tries to stream premium content "m002"
except PermissionError as e:  # Expect a PermissionError for access restriction
    print(f"   Premium blocked for Standard user: {e}")  # Print confirmation of access block

# Example 2 — Noura (Premium) streams premium content
u2 = RegularUser("u006", "Noura Al Shamsi",
                 "noura.alshamsi@streamflix.ae", "pw_006")  # Create a RegularUser object for Noura
u2.subscribe(Subscription("sub-011", premium_plan))  # Subscribe Noura to Premium plan
result2 = service.stream_content(u2, "m002")  # Stream premium content "m002" for Noura
print(f"\nExample 2: {result2}")  # Print streaming result
assert "Now streaming" in result2  # Verify streaming was successful
print("   Noura streams Dune Part Two (premium content).")  # Print success message

# Unavailable content blocked
cat.update_availability("t001", False)  # Mark content "t001" as unavailable
try:
    service.stream_content(u2, "t001")  # Noura attempts to stream unavailable content
except PermissionError as e:  # Expect a PermissionError due to unavailability
    print(f"   Unavailable content blocked: {e}")  # Print confirmation of block
cat.update_availability("t001", True)  # Restore content "t001" availability

print("\nTEST 4 PASSED ")  # Print final confirmation that Test 4 passed


──────────────────────────────────────────────────────────
TEST 4: Streaming Content Access
  [Catalogue] Added: 'Inception'
  [Catalogue] Added: 'Dune: Part Two'
  [Catalogue] Added: 'Breaking Bad'
  [Catalogue] Added: 'House of Dragon'
  [Catalogue] Added: 'Planet Earth III'

Example 1: Now streaming: 'Inception' (Movie) for Khalid Al Mansoori
   Khalid streams Inception (free content).
   Premium blocked for Standard user: Access denied. Upgrade plan or purchase 'Dune: Part Two' to stream.

Example 2: Now streaming: 'Dune: Part Two' (Movie) for Noura Al Shamsi
   Noura streams Dune Part Two (premium content).
  [Catalogue] 'Breaking Bad' marked unavailable.
   Unavailable content blocked: 'Breaking Bad' is currently unavailable.
  [Catalogue] 'Breaking Bad' marked available.

TEST 4 PASSED 


In [72]:
print(SEP + "TEST 5: Payment Processing (Currency: AED)")  # Print a header for Test 5 with separator

_, _, premium_plan = make_plans()  # Create subscription plans; ignore Basic and Standard, keep Premium
cat     = make_catalogue()  # Create a content catalogue with movies and shows
service = StreamingService(cat)  # Initialize streaming service with the catalogue

# Example 1 — Ahmed Al Hashimi purchases Dune via Credit Card
u1 = RegularUser("u007", "Ahmed Al Hashimi",
                 "ahmed.alhashimi@streamflix.ae", "pw_007")  # Create RegularUser object for Ahmed
u1.subscribe(Subscription("sub-020", premium_plan))  # Subscribe Ahmed to Premium plan

inv1 = service.purchase_content(u1, "m002", PaymentMethod.CREDIT_CARD)  # Ahmed purchases premium content "m002" via Credit Card

print("\nExample 1 — Ahmed Al Hashimi purchases 'Dune: Part Two' via Credit Card:")  # Print example description
print("  Customer    : Ahmed Al Hashimi")  # Print customer name
print("  Email       : ahmed.alhashimi@streamflix.ae")  # Print customer email
print("  Content     : Dune: Part Two (Premium Movie)")  # Print content purchased
print("  Method      : Credit Card")  # Print payment method
print("  Amount      : AED " + str(round(inv1.get_subtotal() * 3.67, 2)))  # Print subtotal in AED (converted from USD)
print("  Tax (10%)   : AED " + str(round(inv1.get_tax_amount() * 3.67, 2)))  # Print tax amount in AED
print("  Total Paid  : AED " + str(round(inv1.get_total() * 3.67, 2)))  # Print total paid in AED
print("  Invoice ID  : " + inv1.get_invoice_id())  # Print invoice ID
print("  Status      : Payment Completed")  # Print payment status

assert inv1.get_total() > 0  # Ensure total payment is positive
assert "m002" in u1.get_purchased_content()  # Verify content is unlocked for the user
print("   Payment completed and Dune unlocked for Ahmed Al Hashimi.")  # Print success message

# Example 2 — Sara Al Nuaimi pays Premium subscription via PayPal with AED 7 discount
u2 = RegularUser("u008", "Sara Al Nuaimi",
                 "sara.alnuaimi@streamflix.ae", "pw_008")  # Create RegularUser object for Sara
u2.subscribe(Subscription("sub-021", premium_plan))  # Subscribe Sara to Premium plan

# discount in USD = 7 AED / 3.67 ≈ 1.91 USD
discount_aed = 7.00  # Discount amount in AED
discount_usd = round(discount_aed / 3.67, 2)  # Convert discount to USD for processing

inv2 = service.process_subscription_payment(u2, PaymentMethod.PAYPAL, discount=discount_usd)
# Process subscription payment via PayPal with USD discount

print("\nExample 2 — Sara Al Nuaimi pays Premium Subscription via PayPal:")  # Print example description
print("  Customer    : Sara Al Nuaimi")  # Print customer name
print("  Email       : sara.alnuaimi@streamflix.ae")  # Print customer email
print("  Plan        : Premium Plan")  # Print subscription plan
print("  Method      : PayPal")  # Print payment method
print("  Subtotal    : AED " + str(round(inv2.get_subtotal() * 3.67, 2)))  # Print subtotal in AED
print("  Discount    : AED " + str(discount_aed))  # Print discount in AED
print("  Tax (10%)   : AED " + str(round(inv2.get_tax_amount() * 3.67, 2)))  # Print tax in AED
print("  Total Paid  : AED " + str(round(inv2.get_total() * 3.67, 2)))  # Print total paid in AED
print("  Invoice ID  : " + inv2.get_invoice_id())  # Print invoice ID
print("  Status      : Payment Completed")  # Print payment status

assert inv2.get_discount() == discount_usd  # Verify discount applied correctly
print("   Subscription payment with AED 7 discount verified for Sara Al Nuaimi.")  # Print success message

# Exception: duplicate purchase
try:
    service.purchase_content(u1, "m002", PaymentMethod.DEBIT_CARD)  # Attempt duplicate purchase for Ahmed
except ValueError as e:  # Expect ValueError for duplicate purchase
    print("\n   Duplicate purchase blocked: " + str(e))  # Print confirmation of blocked duplicate

print("\nTEST 5 PASSED ")  # Print final confirmation that Test 5 passed


──────────────────────────────────────────────────────────
TEST 5: Payment Processing (Currency: AED)
  [Catalogue] Added: 'Inception'
  [Catalogue] Added: 'Dune: Part Two'
  [Catalogue] Added: 'Breaking Bad'
  [Catalogue] Added: 'House of Dragon'
  [Catalogue] Added: 'Planet Earth III'

Example 1 — Ahmed Al Hashimi purchases 'Dune: Part Two' via Credit Card:
  Customer    : Ahmed Al Hashimi
  Email       : ahmed.alhashimi@streamflix.ae
  Content     : Dune: Part Two (Premium Movie)
  Method      : Credit Card
  Amount      : AED 21.98
  Tax (10%)   : AED 2.2
  Total Paid  : AED 24.19
  Invoice ID  : inv-2001
  Status      : Payment Completed
   Payment completed and Dune unlocked for Ahmed Al Hashimi.

Example 2 — Sara Al Nuaimi pays Premium Subscription via PayPal:
  Customer    : Sara Al Nuaimi
  Email       : sara.alnuaimi@streamflix.ae
  Plan        : Premium Plan
  Method      : PayPal
  Subtotal    : AED 66.02
  Discount    : AED 7.0
  Tax (10%)   : AED 5.91
  Total Paid  : AED

In [73]:
print(SEP + "TEST 6: Invoice Generation (Currency: AED)")  # Print header for Test 6 with separator

# Conversion rate
AED_RATE = 3.67   # 1 USD = 3.67 AED

def usd_to_aed(amount):
    return round(amount * AED_RATE, 2)  # Convert USD amount to AED and round to 2 decimals

# ── Example 1 — Hassan Al Blooshi, Premium Plan, default 10% VAT ─────────────
# UAE uses VAT = 5%, but system default is 10%
# Premium plan = $17.99 USD ≈ AED 66.02

subtotal_usd_1 = 17.99  # Subtotal in USD for Premium subscription
inv1 = Invoice(
    "inv-6001",
    subtotal_usd_1,
    payment_id  = "pay-6001",
    description = "Subscription: Premium Plan"
)  # Create Invoice object for Hassan

subtotal_aed_1 = usd_to_aed(inv1.get_subtotal())  # Convert subtotal to AED
tax_aed_1      = usd_to_aed(inv1.get_tax_amount())  # Convert tax amount to AED
total_aed_1    = usd_to_aed(inv1.get_total())  # Convert total amount to AED

print("\nExample 1 — Invoice for Hassan Al Blooshi (Premium Plan, 10% Tax):")  # Print example description
print("  ╔══════════════════════════════════════════════╗")  # Invoice top border
print("  ║       STREAMFLIX INVOICE  #inv-6001          ║")  # Invoice number
print("  ╠══════════════════════════════════════════════╣")  # Separator
print("  ║  Customer  : Hassan Al Blooshi               ║")  # Customer name
print("  ║  Email     : hassan.alblooshi@streamflix.ae  ║")  # Customer email
print("  ║  Plan      : Premium Subscription            ║")  # Plan description
print("  ║  Invoice ID: inv-6001                        ║")  # Invoice ID
print("  ║  Payment   : pay-6001                        ║")  # Payment ID
print("  ╠══════════════════════════════════════════════╣")  # Separator
print("  ║  Subtotal  :             AED " + str(subtotal_aed_1).ljust(10) + "      ║")  # Subtotal in AED
print("  ║  Discount  :             AED 0.00            ║")  # Discount line (0 for subscription)
print("  ║  Tax (10%) :             AED " + str(tax_aed_1).ljust(10) + "      ║")  # Tax in AED
print("  ╠══════════════════════════════════════════════╣")  # Separator
print("  ║  TOTAL     :             AED " + str(total_aed_1).ljust(10) + "      ║")  # Total in AED
print("  ╚══════════════════════════════════════════════╝")  # Bottom border

assert inv1.get_tax_amount() == round(subtotal_usd_1 * 0.10, 2)  # Verify tax amount
assert inv1.get_total()      == round(subtotal_usd_1 * 1.10, 2)  # Verify total amount
print("  Tax = AED " + str(tax_aed_1) + " | Total = AED " + str(total_aed_1) + " verified.")  # Success message

# ── Example 2 — Maryam Al Ketbi, Dune Purchase, AED 5 discount, 5% VAT ──────
# Content price = $5.99 USD ≈ AED 21.98
# Discount = AED 5 → $1.36 USD
# UAE VAT = 5%

subtotal_usd_2  = 5.99  # Subtotal in USD for content purchase
discount_aed_2  = 5.00  # Discount in AED
discount_usd_2  = round(discount_aed_2 / AED_RATE, 2)  # Convert discount to USD
vat_rate        = 0.05   # Tax rate 5% for UAE VAT

inv2 = Invoice(
    "inv-6002",
    subtotal_usd_2,
    discount = discount_usd_2,
    tax_rate = vat_rate,
    payment_id  = "pay-6002",
    description = "Purchase: Dune Part Two"
)  # Create Invoice object for Maryam with discount and VAT

subtotal_aed_2  = usd_to_aed(inv2.get_subtotal())  # Convert subtotal to AED
discount_show   = discount_aed_2  # Discount to display in AED
tax_aed_2       = usd_to_aed(inv2.get_tax_amount())  # Convert tax amount to AED
total_aed_2     = usd_to_aed(inv2.get_total())  # Convert total amount to AED

print("\nExample 2 — Invoice for Maryam Al Ketbi (Dune Purchase, AED 5 Discount, 5% VAT):")  # Print example description
print("  ╔══════════════════════════════════════════════╗")  # Invoice top border
print("  ║       STREAMFLIX INVOICE  #inv-6002          ║")  # Invoice number
print("  ╠══════════════════════════════════════════════╣")  # Separator
print("  ║  Customer  : Maryam Al Ketbi                 ║")  # Customer name
print("  ║  Email     : maryam.alketbi@streamflix.ae    ║")  # Customer email
print("  ║  Content   : Dune: Part Two (Premium Movie)  ║")  # Purchased content
print("  ║  Invoice ID: inv-6002                        ║")  # Invoice ID
print("  ║  Payment   : pay-6002                        ║")  # Payment ID
print("  ╠══════════════════════════════════════════════╣")  # Separator
print("  ║  Subtotal  :             AED " + str(subtotal_aed_2).ljust(10) + "      ║")  # Subtotal
print("  ║  Discount  :             AED " + str(discount_show).ljust(10) + "      ║")  # Discount line
print("  ║  VAT (5%)  :             AED " + str(tax_aed_2).ljust(10) + "      ║")  # Tax line
print("  ╠══════════════════════════════════════════════╣")  # Separator
print("  ║  TOTAL     :             AED " + str(total_aed_2).ljust(10) + "      ║")  # Total line
print("  ╚══════════════════════════════════════════════╝")  # Bottom border

assert inv2.get_tax_rate()  == 0.05  # Verify VAT rate
assert inv2.get_discount()  == discount_usd_2  # Verify discount applied
assert inv2.get_total()     > 0  # Ensure total is positive
print("  ✓ VAT = AED " + str(tax_aed_2) + " | Total = AED " + str(total_aed_2) + " verified.")  # Success message

# ── Exception: Invalid tax rate ──────────────────────────────────────────────
try:
    inv1.set_tax_rate(2.5)  # Attempt to set invalid tax rate
except ValueError as e:  # Expect ValueError
    print("\n  ✓ Caught invalid tax rate: " + str(e))  # Print exception message

# ── Exception: Negative discount ─────────────────────────────────────────────
try:
    inv2.set_discount(-10.0)  # Attempt to set negative discount
except ValueError as e:  # Expect ValueError
    print("  ✓ Caught negative discount: " + str(e))  # Print exception message

print("\nTEST 6 PASSED ")  # Print final confirmation that Test 6 passed


──────────────────────────────────────────────────────────
TEST 6: Invoice Generation (Currency: AED)

Example 1 — Invoice for Hassan Al Blooshi (Premium Plan, 10% Tax):
  ╔══════════════════════════════════════════════╗
  ║       STREAMFLIX INVOICE  #inv-6001          ║
  ╠══════════════════════════════════════════════╣
  ║  Customer  : Hassan Al Blooshi               ║
  ║  Email     : hassan.alblooshi@streamflix.ae  ║
  ║  Plan      : Premium Subscription            ║
  ║  Invoice ID: inv-6001                        ║
  ║  Payment   : pay-6001                        ║
  ╠══════════════════════════════════════════════╣
  ║  Subtotal  :             AED 66.02           ║
  ║  Discount  :             AED 0.00            ║
  ║  Tax (10%) :             AED 6.61            ║
  ╠══════════════════════════════════════════════╣
  ║  TOTAL     :             AED 72.63           ║
  ╚══════════════════════════════════════════════╝
  Tax = AED 6.61 | Total = AED 72.63 verified.

Example 2 — Invo

In [74]:
print(SEP + "TEST 7: Viewing History Tracking")  # Print header for Test 7 with separator

_, standard, _ = make_plans()  # Create subscription plans; keep Standard plan only
cat     = make_catalogue()  # Create content catalogue
service = StreamingService(cat)  # Initialize streaming service with the catalogue

# Example 1 — Rashid watches two titles
u1 = RegularUser("u009", "Rashid Al Muhairi",
                 "rashid.almuhairi@streamflix.ae", "pw_009")  # Create RegularUser object for Rashid
u1.subscribe(Subscription("sub-040", standard))  # Subscribe Rashid to Standard plan
service.stream_content(u1, "m001")  # Rashid streams "m001" (Inception)
service.stream_content(u1, "t001")  # Rashid streams "t001" (Breaking Bad)
history = u1.get_viewing_history()  # Retrieve viewing history for Rashid
print("\nExample 1 — Rashid Al Muhairi's viewing history:")  # Print example description
print(u1.display_viewing_history())  # Display formatted viewing history
assert len(history) == 2  # Ensure two records are stored
assert history[0].get_content_title() == "Inception"  # Verify first content title
assert history[1].get_content_title() == "Breaking Bad"  # Verify second content title
print("  Two records stored and retrieved correctly.")  # Print success message

# Example 2 — Layla adds manual record and updates progress
u2 = RegularUser("u010", "Layla Al Hammadi",
                 "layla.alhammadi@streamflix.ae", "pw_010")  # Create RegularUser object for Layla
rec = ViewingRecord("rec-m1", "d001", "Planet Earth III", progress_pct=40.0)  # Create a manual viewing record
u2.add_viewing_record(rec)  # Add the viewing record to Layla's history
print(f"\nExample 2 — Layla Al Hammadi's record: {rec}")  # Print the record details
rec.set_progress_pct(75.0)  # Update progress percentage to 75%
assert rec.get_progress_pct() == 75.0  # Verify updated progress
print("   Progress updated to 75%.")  # Print success message

# Exception: invalid progress
try:
    rec.set_progress_pct(120.0)  # Attempt to set invalid progress >100%
except ValueError as e:  # Expect ValueError
    print(f"  Caught invalid progress: {e}")  # Print exception confirmation

print("\nTEST 7 PASSED ")  # Print final confirmation that Test 7 passed


──────────────────────────────────────────────────────────
TEST 7: Viewing History Tracking
  [Catalogue] Added: 'Inception'
  [Catalogue] Added: 'Dune: Part Two'
  [Catalogue] Added: 'Breaking Bad'
  [Catalogue] Added: 'House of Dragon'
  [Catalogue] Added: 'Planet Earth III'

Example 1 — Rashid Al Muhairi's viewing history:
Viewing History:
  [2026-03-29] Inception — 100% watched
  [2026-03-29] Breaking Bad — 100% watched
  Two records stored and retrieved correctly.

Example 2 — Layla Al Hammadi's record: [2026-03-29] Planet Earth III — 40% watched
   Progress updated to 75%.
  Caught invalid progress: Progress must be between 0 and 100.

TEST 7 PASSED 


In [75]:
print(SEP + "TEST 8: Notification System")  # Print header for Test 8 with separator

basic, _, premium_plan = make_plans()  # Create subscription plans; keep Basic and Premium plans

# Example 1 — Zayed Al Nahyan gets subscription notification
u1 = RegularUser("u011", "Zayed Al Nahyan",
                 "zayed.alnahyan@streamflix.ae", "pw_011")  # Create RegularUser object for Zayed
u1.subscribe(Subscription("sub-050", basic))  # Subscribe Zayed to Basic plan
unread = u1.get_unread_notifications()  # Get unread notifications after subscription
print("\nExample 1 — Zayed Al Nahyan notifications after subscribe:")  # Print example description
for n in unread:  # Loop through unread notifications
    print(f"  {n}")  # Print each notification
assert len(unread) >= 1  # Ensure at least one notification exists
assert "Basic" in unread[0].get_message()  # Verify notification message contains plan name
u1.mark_all_notifications_read()  # Mark all notifications as read
assert len(u1.get_unread_notifications()) == 0  # Verify no unread notifications remain
print("  Notification received and marked read.")  # Print success message

# Example 2 — Hessa Al Maktoum gets payment notification
cat     = make_catalogue()  # Create content catalogue
service = StreamingService(cat)  # Initialize streaming service
u2 = RegularUser("u012", "Hessa Al Maktoum",
                 "hessa.almaktoum@streamflix.ae", "pw_012")  # Create RegularUser object for Hessa
u2.subscribe(Subscription("sub-051", premium_plan))  # Subscribe Hessa to Premium plan
u2.mark_all_notifications_read()  # Clear any previous notifications
service.process_subscription_payment(u2, PaymentMethod.CREDIT_CARD)  # Process subscription payment
unread2 = u2.get_unread_notifications()  # Get unread notifications after payment
print("\nExample 2 — Hessa Al Maktoum notifications after payment:")  # Print example description
for n in unread2:  # Loop through unread notifications
    print(f"  {n}")  # Print each notification
assert any(n.get_notif_type() == "payment" for n in unread2)  # Ensure at least one payment notification exists
print("   Payment notification received.")  # Print success message

# Content availability notification (manual)
u2.add_notification(Notification("n-c1", "New: 'Avatar 3' is now available!", "content"))  # Manually add content notification
content_notifs = [n for n in u2.get_unread_notifications() if n.get_notif_type() == "content"]  # Filter for content notifications
assert len(content_notifs) == 1  # Ensure one content notification exists
print(f"   Content notification: {content_notifs[0].get_message()}")  # Print content notification message

print("\nTEST 8 PASSED ")  # Print final confirmation that Test 8 passed


──────────────────────────────────────────────────────────
TEST 8: Notification System

Example 1 — Zayed Al Nahyan notifications after subscribe:
  [Unread][SUBSCRIPTION] 2026-03-29 11:50 — You subscribed to the Basic plan.
  Notification received and marked read.
  [Catalogue] Added: 'Inception'
  [Catalogue] Added: 'Dune: Part Two'
  [Catalogue] Added: 'Breaking Bad'
  [Catalogue] Added: 'House of Dragon'
  [Catalogue] Added: 'Planet Earth III'

Example 2 — Hessa Al Maktoum notifications after payment:
  [Unread][PAYMENT] 2026-03-29 11:50 — Subscription renewed. Invoice #inv-2001 total: $19.79.
   Payment notification received.
   Content notification: New: 'Avatar 3' is now available!

TEST 8 PASSED 


In [76]:
print(SEP + "TEST 9: Administrator Actions")  # Print header for Test 9 with separator

admin = Administrator("a001", "Amira Hassan",
                      "amira.hassan@streamflix.ae",
                      "admin_001", admin_level=3, department="Operations")  # Create Administrator object for Amira
user  = RegularUser("u001", "Mohammed Al Rashid",
                    "mohammed.alrashid@streamflix.ae", "pw_001")  # Create RegularUser object for Mohammed
cat   = make_catalogue()  # Create content catalogue

# Reset password
admin.reset_user_password(user, "new_hash_2026")  # Admin resets user's password
assert user.get_password_hash() == "new_hash_2026"  # Verify password hash updated
print("\n   Amira reset Mohammed Al Rashid's password.")  # Print confirmation

# Deactivate account
admin.deactivate_user(user)  # Admin deactivates user account
assert user.get_is_active() is False  # Verify account is inactive
print("   Amira deactivated Mohammed Al Rashid's account.")  # Print confirmation

# Toggle content availability
content  = cat.get_content_by_id("m001")  # Retrieve content object "m001" (Inception)
original = content.get_is_available()  # Store original availability status
admin.toggle_content_availability(content)  # Admin toggles availability
assert content.get_is_available() != original  # Verify availability has changed
print(f"   'Inception' availability toggled to {content.get_is_available()}.")  # Print new status

# Feedback submission
_, standard, _ = make_plans()  # Create plans; keep Standard plan only
u2  = RegularUser("u005", "Khalid Al Mansoori",
                  "khalid.almansoori@streamflix.ae", "pw_005")  # Create RegularUser object for Khalid
u2.subscribe(Subscription("sub-060", standard))  # Subscribe Khalid to Standard plan
fb  = Feedback("fb-001", "m001", rating=5, comment="Absolute masterpiece!")  # Create Feedback object
u2.submit_feedback(fb)  # User submits feedback
assert len(u2.get_feedback_list()) == 1  # Verify feedback recorded
print(f"   Feedback by Khalid: {u2.get_feedback_list()[0]}")  # Print feedback confirmation

print("\nTEST 9 PASSED ")  # Print final confirmation that Test 9 passed


──────────────────────────────────────────────────────────
TEST 9: Administrator Actions
  [Catalogue] Added: 'Inception'
  [Catalogue] Added: 'Dune: Part Two'
  [Catalogue] Added: 'Breaking Bad'
  [Catalogue] Added: 'House of Dragon'
  [Catalogue] Added: 'Planet Earth III'

   Amira reset Mohammed Al Rashid's password.
   Amira deactivated Mohammed Al Rashid's account.
   'Inception' availability toggled to False.
   Feedback by Khalid: [Feedback fb-001] ★★★★★ — Absolute masterpiece!

TEST 9 PASSED 


In [59]:
print("\n" + "═" * 58)
print("   STREAMFLIX — COMPLETE TEST RESULTS")
print("═" * 58)

all_tests = [
    ("Test 1 — User Account Creation",      test_user_account_creation    if 'test_user_account_creation' in dir() else None),
    ("Test 2 — Content Catalogue",          test_content_catalogue        if 'test_content_catalogue' in dir() else None),
    ("Test 3 — Subscription Management",    test_subscription_management  if 'test_subscription_management' in dir() else None),
    ("Test 4 — Streaming Access",           test_streaming_access         if 'test_streaming_access' in dir() else None),
    ("Test 5 — Payment Processing",         test_payment_processing       if 'test_payment_processing' in dir() else None),
    ("Test 6 — Invoice Generation",         test_invoice_generation       if 'test_invoice_generation' in dir() else None),
    ("Test 7 — Viewing History",            test_viewing_history          if 'test_viewing_history' in dir() else None),
    ("Test 8 — Notification System",        test_notifications            if 'test_notifications' in dir() else None),
    ("Test 9 — Administrator Actions",      test_admin_actions            if 'test_admin_actions' in dir() else None),
]

print("\nAll tests ran individually above — each cell shows PASSED ✓")
print("\nSummary:")
print("   Test 1 — User Account Creation")
print("   Test 2 — Content Catalogue Management")
print("   Test 3 — Subscription Plan Management")
print("   Test 4 — Streaming Content Access")
print("   Test 5 — Payment Processing")
print("   Test 6 — Invoice Generation")
print("   Test 7 — Viewing History Tracking")
print("   Test 8 — Notification System")
print("   Test 9 — Administrator Actions")
print("\n" + "═" * 58)
print("  9/9 TESTS PASSED ")
print("═" * 58)


══════════════════════════════════════════════════════════
   STREAMFLIX — COMPLETE TEST RESULTS
══════════════════════════════════════════════════════════

All tests ran individually above — each cell shows PASSED ✓

Summary:
   Test 1 — User Account Creation
   Test 2 — Content Catalogue Management
   Test 3 — Subscription Plan Management
   Test 4 — Streaming Content Access
   Test 5 — Payment Processing
   Test 6 — Invoice Generation
   Test 7 — Viewing History Tracking
   Test 8 — Notification System
   Test 9 — Administrator Actions

══════════════════════════════════════════════════════════
  9/9 TESTS PASSED 
══════════════════════════════════════════════════════════
